# Train the thermal CNN-LSTM on a Colab GPU

Runs `05_train_cnn_lstm.py` and `06_evaluate_test.py` on a free Colab GPU instead of a laptop CPU.

**Before you start**
1. In Colab: **Runtime → Change runtime type → T4 GPU**.
2. Upload your local `Data\sequences` folder (the one `04_build_sequences.py` made, with `train/`, `val/`, `test/` inside) to Google Drive as
   `My Drive/thermal_behaviour/sequences`.

Then run the cells top to bottom. The trained model and the test report are saved to `My Drive/thermal_behaviour/` as training goes, so a disconnect doesn't lose the best checkpoint.

In [ ]:
# 1. Check the GPU is on
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print("TensorFlow", tf.__version__, "| GPUs:", gpus)
assert gpus, "No GPU: Runtime -> Change runtime type -> T4 GPU, then run this cell again."

In [ ]:
# 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Folder where models/ and outputs/ are saved. The sequences can be anywhere on your Drive.
DRIVE_DIR = '/content/drive/MyDrive/thermal_behaviour'
import os
os.makedirs(DRIVE_DIR, exist_ok=True)
print("Results will be saved to:", DRIVE_DIR)

In [ ]:
# 3. Get the scripts
!rm -rf /content/repo
!git clone -q -b claude/gracious-dirac-kdb1q7 https://github.com/abhinaya-thota/thermal-behavior-recognition.git /content/repo
!ls /content/repo

In [ ]:
# 4. Find the sequences on Drive, copy them to the Colab machine's local disk
#    (much faster to read than Drive), and send models/ and outputs/ straight to Drive.
import glob, shutil
WORK = '/content/work'

hits = sorted(glob.glob('/content/drive/MyDrive/**/train/labels.npy', recursive=True))
hits = [h for h in hits if all(os.path.exists(os.path.join(os.path.dirname(os.path.dirname(h)), s, 'X.npy'))
                               for s in ['train', 'val', 'test'])]
if not hits:
    raise SystemExit(
        "STOP: no complete sequences folder (train/, val/, test/ with X.npy and labels.npy) found on your Drive.\n"
        "The upload is probably not finished. Check the upload box on drive.google.com, wait for the green ticks,\n"
        "then run this cell again. Do NOT run training (cell 6) until this cell prints 'Ready'.")
SEQ_DIR = os.path.dirname(os.path.dirname(hits[0]))
if len(hits) > 1:
    print("Several sequences folders found, using the first:", [os.path.dirname(os.path.dirname(h)) for h in hits])
print("Using sequences from:", SEQ_DIR)

shutil.rmtree(WORK, ignore_errors=True)
os.makedirs(f'{WORK}/data')
shutil.copytree(SEQ_DIR, f'{WORK}/data/sequences')
for s in ['train', 'val', 'test']:
    files = sorted(os.listdir(f'{WORK}/data/sequences/{s}'))
    gb = sum(os.path.getsize(f'{WORK}/data/sequences/{s}/{f}') for f in files) / 1e9
    print(f'{s:5s} {gb:5.2f} GB  {files}')

for d in ['models', 'outputs']:
    os.makedirs(os.path.join(DRIVE_DIR, d), exist_ok=True)
    os.symlink(os.path.join(DRIVE_DIR, d), os.path.join(WORK, d))
print("models/ and outputs/ ->", DRIVE_DIR)
print("\nReady: run cells 5-8.")

In [ ]:
# 5. Settings for the GPU. Batch 32 runs out of memory on a T4 (15 GB) with 16x112x112 clips; 16 fits.
#    If you still get RESOURCE_EXHAUSTED / OOM, set this to 8 and re-run this cell and the next.
BATCH_SIZE = 16

import re
src = open('/content/repo/05_train_cnn_lstm.py').read()
src = re.sub(r'^BATCH_SIZE = \d+', f'BATCH_SIZE = {BATCH_SIZE}', src, flags=re.M)
open('/content/repo/05_train_cnn_lstm.py', 'w').write(src)
!grep -n "^BATCH_SIZE" /content/repo/05_train_cnn_lstm.py

In [ ]:
# 6. Train (the best model is saved to Drive after every improving epoch)
%cd {WORK}
!python /content/repo/05_train_cnn_lstm.py

In [ ]:
# 7. Evaluate on the test set
%cd {WORK}
!python /content/repo/06_evaluate_test.py

In [ ]:
# 8. Show the results (also saved in My Drive/thermal_behaviour/outputs/)
print(open(f'{WORK}/outputs/test_report.txt').read())
from IPython.display import Image, display
for f in ['training_accuracy.png', 'confusion_matrix_test.png']:
    p = f'{WORK}/outputs/{f}'
    if os.path.exists(p):
        display(Image(p))

## Upgrade: pretrained backbone + augmentation (`05b_train_pretrained.py`)

The from-scratch CNN-LSTM memorises training backgrounds (train ~96%, test ~27%). This version uses MobileNetV2 pretrained on ImageNet for each frame, augments clips during training, and fine-tunes in two phases. It uses the same data, so **run cells 1–4 first** (or just keep this runtime), then these cells. The old model is kept for comparison.

In [ ]:
# 9. Train the pretrained-backbone model (saved to Drive: models/thermal_mobilenet_lstm.keras)
%cd {WORK}
!python /content/repo/05b_train_pretrained.py

In [ ]:
# 10. Evaluate it on the test set
%cd {WORK}
!python /content/repo/06_evaluate_test.py --model models/thermal_mobilenet_lstm.keras

In [ ]:
# 11. Show the results next to the old model's
for name in ['test_report.txt', 'test_report_thermal_mobilenet_lstm.txt']:
    p = f'{WORK}/outputs/{name}'
    if os.path.exists(p):
        print(f'===== {name} =====')
        print(open(p).read())
from IPython.display import Image, display
for f in ['training_accuracy_pretrained.png', 'confusion_matrix_test_thermal_mobilenet_lstm.png']:
    p = f'{WORK}/outputs/{f}'
    if os.path.exists(p):
        display(Image(p))